# 05. Baseline ML trên đặc trưng node (task 4.1)

Đọc đồ thị từ output của kernel `gnn-iot-ids-04-build-graphs`. Chọn cấu hình trên **val** (seed 42), rồi chạy 3 seed 42, 43, 44; chỉ ghi `split=val` (test chỉ dùng 1 lần ở GĐ6). F1-macro tính trên 4 lớp chính (D5). Độ trễ đo trên 1 luồng với 1 cửa sổ có số node trung vị.

In [ ]:
import glob, os, subprocess, sys

def run(cmd, ok=(0,), **kw):
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    print(r.stdout[-8000:], end="")
    print(r.stderr[-2500:], end="", file=sys.stderr)
    print(f"[mã thoát {r.returncode}]")
    if r.returncode not in ok:
        raise RuntimeError(f"Lệnh lỗi: {' '.join(cmd)}")

REPO = "/tmp/gnn-iot-ids"
run(["rm", "-rf", REPO])
run(["git", "clone", "--depth", "1", "https://github.com/minucedev/gnn-iot-ids.git", REPO])
run(["git", "-C", REPO, "log", "-1", "--format=%H %s"])
with open(f"{REPO}/requirements.txt") as f:
    reqs = [l.strip() for l in f if l.strip() and not l.startswith("#")]
keep = [r for r in reqs if r.split("==")[0].lower() not in {"torch", "jupyter", "ipykernel"}]
run([sys.executable, "-m", "pip", "install", "-q", *keep])
print(os.cpu_count(), "CPU")

In [ ]:
OUT = "/kaggle/working"
for ds in ("nf_ton_iot_v3", "nf_cicids2018_v3", "nf_ton_iot_v2"):
    d = os.path.dirname(next(glob.iglob(f"/kaggle/input/**/{ds}/split.json", recursive=True)))
    print("=====", ds, d)
    # kiểm tra lại đồ thị đã lưu (mã 3 = có kiểm tra không đạt: vẫn chạy tiếp để thấy hết, ghi vào verify_*.json)
    run([sys.executable, "-I", "scripts/verify_graphs.py", d, f"{OUT}/verify_{ds}.json"], ok=(0, 3), cwd=REPO)
    run([sys.executable, "-I", "scripts/run_baselines.py", d, ds,
         f"{OUT}/results_baselines_{ds}.csv", f"{OUT}/baseline_search_{ds}.json"], cwd=REPO)